# HI5312 | Python Week 2 — From SQL to Clinical Insight
**Companion to the 27-slide PowerPoint**. Each section identifies its corresponding slide. Run in order in the course Codespace. The examples follow the presentation, with small execution safeguards clearly identified.

## Slides 1–3 — Roadmap and Python refresher
**Discuss:** How can code turn clinical records into meaningful evidence? Variables, DataFrames, functions, libraries, notebooks.

## Slide 4 — Predict the output
**Discuss before running:** What prints? What type is `total`?

In [ ]:
patient_count = 120
new_patients = 15
total = patient_count + new_patients
print(f"Total patients: {total}")
print("Data type:", type(total).__name__)

## Slide 5 — Python toolkit
**Discuss:** Which import handles tables, charts, and databases?

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
try:
    from sqlalchemy import create_engine, text
    from sqlalchemy.engine import URL
    SQLALCHEMY_INSTALLED = True
except ImportError:
    SQLALCHEMY_INSTALLED = False
    print("SQLAlchemy or its dependencies are missing; pandas demonstrations can continue.")

## Slide 6 — Units of analysis
`patients`: one row per `subject_id`; `admissions`: one row per `hadm_id`; `icustays`: one row per `stay_id`. **Discuss:** Why is one row not necessarily one patient?

## Slide 7 — Locate MIMIC-IV Demo
The slide uses `/workspaces/hi5304-notebooks/data/mimic-iv-demo`. This notebook also checks a few standard Codespaces locations and accepts `.csv` or `.csv.gz`.

In [ ]:
bases = [
    Path("/workspaces/hi5304-notebooks/data/mimic-iv-demo"),
    Path("/workspaces/hi5304-notebooks/data/mimic-iv-demo/1.0"),
    Path.cwd() / "data/mimic-iv-demo",
]
base = next((b for b in bases if any((b / "hosp" / ("patients" + ext)).is_file() for ext in (".csv", ".csv.gz"))), None)
if base is None:
    raise FileNotFoundError("MIMIC-IV Demo patients.csv or patients.csv.gz was not found. Place the demo in /workspaces/hi5304-notebooks/data/mimic-iv-demo/hosp or edit bases above.")
hosp, icu = base / "hosp", base / "icu"
print("MIMIC-IV Demo directory:", base)
def read_mimic(folder, name):
    for ext in (".csv", ".csv.gz"):
        path = folder / (name + ext)
        if path.exists():
            return pd.read_csv(path, low_memory=False)
    raise FileNotFoundError(f"Missing {name}.csv or {name}.csv.gz in {folder}")

## Slide 8 — Load, inspect, question
**Discuss:** What should we inspect before trusting a dataset?

In [ ]:
patients = read_mimic(hosp, "patients")
admissions = read_mimic(hosp, "admissions")
icu_stays = read_mimic(icu, "icustays")
print("Patients / admissions:", patients.shape, admissions.shape)
display(admissions.head())
print(admissions.columns.tolist())

## Slide 9 — Filtering: SQL `WHERE` vs pandas
**Discuss:** What changes with `!=`?

In [ ]:
urgent = admissions.loc[admissions["admission_type"] == "URGENT"].copy()
print("Urgent admissions:", len(urgent))
display(urgent[["subject_id", "hadm_id", "admission_type"]].head())

## Slide 10 — `if / elif / else`
**Predict:** What labels do 3, 7, and 8 days receive?

In [ ]:
def label_los(days):
    if pd.isna(days):
        return "Missing"
    elif days < 3:
        return "Short"
    elif days <= 7:
        return "Moderate"
    else:
        return "Long"
for days in (3, 7, 8):
    print(days, "days ->", label_los(days))

## Slide 11 — Length of stay from timestamps
**Discuss:** Why `errors="coerce"`? What should we do with negative stays?

In [ ]:
admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")
admissions["los_days"] = (admissions["dischtime"] - admissions["admittime"]).dt.total_seconds() / 86400
display(admissions[["hadm_id", "admittime", "dischtime", "los_days"]].head())

## Slide 12 — Apply conditional labels and validate
The slide applies `label_los` directly, then checks negative LOS. **Teaching note:** Negative LOS would incorrectly be labeled `Short`; flag it before interpretation.

In [ ]:
admissions["los_group"] = admissions["los_days"].apply(label_los)
print(admissions["los_group"].value_counts(dropna=False))
print("Negative LOS:", (admissions["los_days"] < 0).sum())
# Safeguard for actual analysis (not shown on slide):
admissions.loc[admissions["los_days"] < 0, "los_group"] = "Invalid (negative)"

## Slide 13 — `for` loops for quality checks
**Predict:** How many iterations? What is `col` on iteration 2?

In [ ]:
columns = ["subject_id", "hadm_id", "admission_type", "insurance"]
for col in columns:
    missing = admissions[col].isna().sum()
    print(f"{col}: {missing} missing")

## Slide 14 — A reusable function
**Discuss:** Where is the conditional expression?

In [ ]:
def missing_report(df, columns):
    for col in columns:
        n = df[col].isna().sum()
        pct = 100 * n / len(df) if len(df) else 0
        print(f"{col}: {n} missing ({pct:.1f}%)")
missing_report(admissions, ["insurance", "admission_type"])

## Slide 15 — Join patients and admissions
**Discuss:** Why `many_to_one`? What if `patients.subject_id` is duplicated?

In [ ]:
joined = admissions.merge(
    patients[["subject_id", "gender"]],
    on="subject_id", how="left", validate="many_to_one", indicator=True
)
print(joined["_merge"].value_counts())

## Slide 16 — Admissions per patient
**Discuss:** Why unique `hadm_id` rather than raw rows?

In [ ]:
per_patient = (admissions.groupby("subject_id")["hadm_id"]
               .nunique().reset_index(name="n_admissions"))
print(per_patient["n_admissions"].describe())
print("Multiple admissions:", (per_patient["n_admissions"] > 1).sum())

## Slide 17 — Visualize the distribution
**Discuss:** What pattern would surprise you? What does this chart *not* tell us?

In [ ]:
(per_patient["n_admissions"].value_counts().sort_index().plot(kind="bar"))
plt.xlabel("Admissions per patient")
plt.ylabel("Number of patients")
plt.title("MIMIC-IV Demo: admissions per patient")
plt.tight_layout()
plt.show()

## Slide 18 — SQLAlchemy connection
**Discuss:** What must be running? PostgreSQL and its Python driver are optional for pandas sections, but required for Slides 19–22. Environment variables are preferred for credentials; do not commit passwords to GitHub.

In [ ]:
engine = None
if SQLALCHEMY_INSTALLED:
    try:
        url = URL.create(
            "postgresql+psycopg",
            username=os.getenv("PGUSER", "hi5304_user"),
            password=os.getenv("PGPASSWORD", "hi5304"),
            host=os.getenv("PGHOST", "db"),
            port=int(os.getenv("PGPORT", "5432")),
            database=os.getenv("PGDATABASE", "hi5304"),
        )
        engine = create_engine(url, pool_pre_ping=True, connect_args={"connect_timeout": 3})
        with engine.connect() as conn:
            print("PostgreSQL connected:", conn.execute(text("SELECT current_database() ")).scalar())
    except Exception as exc:
        print("PostgreSQL unavailable; SQL examples will be skipped:", type(exc).__name__, str(exc).splitlines()[0])
        engine = None
else:
    print("SQLAlchemy unavailable; SQL examples will be skipped.")

## Slide 19 — Inspect database schemas and tables
**Discuss:** Why not assume everyone has the same schema?

In [ ]:
if engine is not None:
    with engine.connect() as conn:
        tables = pd.read_sql_query(text("""
            SELECT table_schema, table_name
            FROM information_schema.tables
            WHERE table_schema IN ('mimiciv_hosp', 'mimiciv_icu', 'hi5312_demo')
            ORDER BY table_schema, table_name
        """), conn)
    display(tables.head(20))
else:
    print("SQL query skipped: PostgreSQL is unavailable.")

## Slide 20 — One question, two languages
SQL below is shown as a **string**, not executable bare SQL in a Python cell. **Discuss:** When would you use each approach?

In [ ]:
sql_per_patient = """
SELECT subject_id, COUNT(DISTINCT hadm_id) AS n_admissions
FROM mimiciv_hosp.admissions
GROUP BY subject_id
ORDER BY n_admissions DESC;
"""
print("SQL:\n", sql_per_patient)
print("pandas equivalent:")
display(admissions.groupby("subject_id")["hadm_id"].nunique().head())
if engine is not None:
    try:
        with engine.connect() as conn:
            display(pd.read_sql_query(text(sql_per_patient), conn).head())
    except Exception as exc:
        print("SQL query could not run (verify mimiciv_hosp schema):", type(exc).__name__, str(exc).splitlines()[0])

## Slide 21 — Execute SQL through SQLAlchemy
**Discuss:** What is the pandas `groupby` equivalent?

In [ ]:
sql = """
SELECT admission_type, COUNT(*) AS n_admissions
FROM mimiciv_hosp.admissions
GROUP BY admission_type
ORDER BY n_admissions DESC
"""
if engine is not None:
    try:
        with engine.connect() as conn:
            sql_result = pd.read_sql_query(text(sql), conn)
        display(sql_result)
    except Exception as exc:
        print("SQL query could not run:", type(exc).__name__, str(exc).splitlines()[0])
else:
    print("SQL query skipped: PostgreSQL is unavailable.")
# pandas equivalent
display(admissions.groupby("admission_type", dropna=False).size().reset_index(name="n_admissions"))

## Slide 22 — SQL `JOIN` vs pandas `merge`
**Discuss:** What is the join key? Why `DISTINCT`?

In [ ]:
sql = """
SELECT p.gender, COUNT(DISTINCT a.hadm_id) AS admissions
FROM mimiciv_hosp.admissions AS a
JOIN mimiciv_hosp.patients AS p ON a.subject_id = p.subject_id
GROUP BY p.gender
ORDER BY admissions DESC
"""
if engine is not None:
    try:
        with engine.connect() as conn:
            display(pd.read_sql_query(text(sql), conn))
    except Exception as exc:
        print("SQL query could not run:", type(exc).__name__, str(exc).splitlines()[0])
else:
    print("SQL query skipped: PostgreSQL is unavailable.")
# pandas counterpart using the earlier validated merge
display(joined.groupby("gender", dropna=False)["hadm_id"].nunique().reset_index(name="admissions"))

## Slide 23 — Length of stay by admission type
**Discuss:** Why report the median and mean? Treat invalid negative stays as missing for this *interpretive* summary.

In [ ]:
analysis_admissions = admissions.copy()
analysis_admissions.loc[analysis_admissions["los_days"] < 0, "los_days"] = float("nan")
summary = (analysis_admissions.groupby("admission_type", dropna=False)
           .agg(n=("hadm_id", "nunique"), mean_los=("los_days", "mean"),
                median_los=("los_days", "median")).reset_index())
display(summary.sort_values("n", ascending=False))

## Slide 24 — Think like an analyst
1. What does a row represent?  2. Did a join duplicate observations?  3. Are missing or implausible values handled?  4. What can we legitimately conclude?

**Pair discussion:** Which validation check would you run first?

## Slide 25 — Individual practice (15 minutes)
Choose **one**: A. Load patients/admissions and show counts; B. Calculate LOS and flag missing/negative values; C. Count admissions per patient and plot; D. If PostgreSQL is ready, run a `GROUP BY` with SQLAlchemy.

Record one result and one question.

In [ ]:
# Student workspace: choose A, B, C, or D and write your code here.


## Slide 26 — Small groups (15 minutes)
**Round 1 (5 min):** Each person shares one result.  **Round 2 (5 min):** Discuss an error or validation concern.  **Round 3 (5 min):** Agree on a clinical question worth investigating.

**Report back:** What did you learn about the data, not just the code?

## Slide 27 — Takeaway
**SQL:** retrieve and aggregate in a database. **pandas:** explore, clean, join, summarize, visualize. **`if/elif/else`:** explicit rules. **`for`:** repeat checks. **Always validate before interpreting.**